# 06 - Silver: clima

Lê os JSONs da Bronze e grava `workspace.silver.weather_hourly` (uma linha por cidade, hora e tipo).

**Etapas:**
1. Transforma os arrays horários da API em linhas
2. Une previsão e histórico (o histórico não tem probabilidade de chuva, então fica nula)
3. Extrai a cidade do nome do arquivo
4. Deduplica: se a mesma hora veio em mais de um arquivo, fica com o mais recente
5. Data Quality: temperatura entre -10 e 50 °C, precipitação maior ou igual a 0, probabilidade entre 0 e 100 e data não nula
6. Registros inválidos vão para `workspace.silver.weather_hourly_rejected` (reescrita a cada execução)
7. `MERGE` na tabela final, então é **idempotente**

**Chave:** `cidade` + `observed_at` + `tipo`
```

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = "/Volumes/workspace/bronze/raw_files/weather"
TABELA = "workspace.silver.weather_hourly"
REJEITADOS = "workspace.silver.weather_hourly_rejected"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA} (
  cidade STRING,
  observed_at TIMESTAMP,
  tipo STRING,
  temperature_c DOUBLE,
  apparent_temperature_c DOUBLE,
  precipitation_mm DOUBLE,
  precipitation_probability DOUBLE,
  ingestion_date DATE,
  source_file STRING,
  updated_at TIMESTAMP
)
""")

In [0]:
def ler_bronze(tipo, variaveis):
    df = (spark.read.option("multiLine", True)
          .json(f"{BRONZE}/{tipo}")
          .withColumn("source_file", F.col("_metadata.file_path")))
    arrays = [F.col(f"hourly.{v}").alias(v) for v in ["time"] + variaveis]
    return (df.select("ingestion_date", "source_file",
                      F.explode(F.arrays_zip(*arrays)).alias("h"))
              .select("ingestion_date", "source_file", "h.*")
              .withColumn("tipo", F.lit(tipo)))

prev = ler_bronze("forecast", ["temperature_2m", "apparent_temperature",
                               "precipitation_probability", "precipitation"])
hist = (ler_bronze("historical", ["temperature_2m", "apparent_temperature", "precipitation"])
        .withColumn("precipitation_probability", F.lit(None).cast("double")))

df = (prev.unionByName(hist)
      .withColumn("cidade", F.regexp_extract("source_file", r"/([a-z_]+)_\d{6}\.json$", 1))
      .select(
          "cidade",
          F.col("time").cast("timestamp").alias("observed_at"),
          "tipo",
          F.col("temperature_2m").cast("double").alias("temperature_c"),
          F.col("apparent_temperature").cast("double").alias("apparent_temperature_c"),
          F.col("precipitation").cast("double").alias("precipitation_mm"),
          F.col("precipitation_probability").cast("double"),
          F.col("ingestion_date").cast("date"),
          "source_file",
      ))

# Se a mesma hora veio em mais de um arquivo, fica com o mais recente
w = (Window.partitionBy("cidade", "observed_at", "tipo")
     .orderBy(F.col("ingestion_date").desc(), F.col("source_file").desc()))
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

print("Linhas após transformação:", df.count())

In [0]:
regra_ok = F.coalesce(
    F.col("observed_at").isNotNull()
    & (F.col("cidade") != "")
    & F.col("temperature_c").between(-10, 50)
    & (F.col("precipitation_mm").isNull() | (F.col("precipitation_mm") >= 0))
    & (F.col("precipitation_probability").isNull()
       | F.col("precipitation_probability").between(0, 100)),
    F.lit(False),
)

validos = df.filter(regra_ok).withColumn("updated_at", F.current_timestamp())
rejeitados = df.filter(~regra_ok).withColumn("rejected_at", F.current_timestamp())

print("Válidos:", validos.count(), "| Rejeitados:", rejeitados.count())

(rejeitados.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJEITADOS))

In [0]:
validos.createOrReplaceTempView("weather_novo")

spark.sql(f"""
MERGE INTO {TABELA} AS t
USING weather_novo AS s
ON  t.cidade = s.cidade
AND t.observed_at = s.observed_at
AND t.tipo = s.tipo
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
display(spark.sql(f"""
SELECT tipo, COUNT(*) AS linhas, MIN(observed_at) AS inicio, MAX(observed_at) AS fim
FROM {TABELA}
GROUP BY tipo
"""))